# Esteira de CI Completa com GitHub Actions — Bella Tavola 🍝
## Testes Automatizados, Integração de Modelo e MLOps (Semanas 3 e 4)

Este caderno consolida as etapas de **Garantia de Qualidade de Código** e **Integração de Modelo Remoto** no pipeline de Integração Contínua (CI).

### Objetivos Práticos:
1. **Testes de API (`pytest`)**: Validar endpoints do FastAPI (`TestClient`), verificando contratos, regras de validação e status codes.
2. **Integração do Registry (`Hugging Face`)**: Baixar de forma segura o modelo `math04cezario/mlops-bella-tavola-v1` usando segredos criptografados do GitHub.
3. **Otimização de Pipeline**: Configurar cache inteligente das dependências do modelo para acelerar o GitHub Actions.
4. **Arquitetura de CI (3 Jobs)**: Entender a divisão lógica entre os ambientes de `Qualidade`, `Integração` e `Relatório`.

--- 
# SEÇÃO 1 — Estrutura de Testes com Pytest & FastAPI TestClient 🧪

Abaixo simulamos a estrutura base do arquivo `tests/conftest.py` e os testes rápidos (*Smoke Tests*) para validar se a aplicação FastAPI inicializa corretamente e expõe as rotas essenciais.

O objetivo é validar a 'saúde' básica da aplicação. Usamos o TestClient para simular um usuário acessando a API. Se esse teste falhar, nem continuamos a esteira, pois significa que a aplicação nem sequer consegue inicializar corretamente.

In [ ]:
!pip install pytest httpx

In [ ]:
!pip install pytest httpx huggingface_hub

In [ ]:
!pip install ipywidgets

In [ ]:
!pip install pytest httpx huggingface_hub scikit-learn==1.6.1

In [27]:
import pytest
from fastapi.testclient import TestClient

# Simulação da Fixture do cliente de teste (Equivalente ao conftest.py)
def criar_client_fake():
    from main import app
    return TestClient(app)

client = criar_client_fake()

print("✅ Estrutura base do TestClient carregada.")

# Exemplo de Smoke Test (Teste de fumaça rápido)
def test_api_health_endpoint():
    # O endpoint /ml/health deve retornar 200 se o modelo estiver em cache
    response = client.get("/ml/health")
    print(f"Status Code retornado: {response.status_code}")
    print(f"Corpo da resposta: {response.json()}")
    assert response.status_code in [200, 503] # Aceita 503 se rodar sem token local

test_api_health_endpoint()

✅ Estrutura base do TestClient carregada.
Status Code retornado: 200
Corpo da resposta: {'api': 'ok', 'model': 'ok', 'model_repo': 'math04cezario/mlops-bella-tavola-v1', 'detail': None}


--- 
# SEÇÃO 2 — Testes Avançados e Validação Parametrizada 📊

Para blindar a API contra quebras de contrato de dados, usamos o `@pytest.mark.parametrize`. Isso garante que requisições com dados fora dos limites de negócio tomem erro **422 Unprocessable Entity** automaticamente, protegendo o modelo.

In [28]:
# Payload base perfeitamente válido para o Bella Tavola
PAYLOAD_VALIDO = {
    "valor_pedido": 120.0,
    "hora_pedido": 20,
    "num_itens": 3,
    "historico_cancelamentos": 0,
    "distancia_entrega": 2.5
}

def testar_casos_invalidos_manualmente():
    # Casos de teste que DEVEM falhar na validação do Pydantic
    casos_teste = [
        ("hora_pedido", 25),        # Hora estrapola o limite de 23
        ("hora_pedido", -1),        # Hora negativa
        ("num_itens", 0),           # Pedidos sem itens
        ("valor_pedido", -10.0),    # Valor de compra negativo
        ("distancia_entrega", -5.0) # Distância negativa
    ]
    
    print("🔍 Validando respostas de erro da API (Esperado: 422):")
    for campo, valor_invalido in casos_teste:
        payload_corrompido = {**PAYLOAD_VALIDO, campo: valor_invalido}
        response = client.post("/ml/predict", json=payload_corrompido)
        print(f" -> Campo [{campo}] com valor [{valor_invalido}] respondeu com: {response.status_code}")
        assert response.status_code == 422
    print("✅ Todos os testes de contrato estático passaram!")

testar_casos_invalidos_manualmente()

🔍 Validando respostas de erro da API (Esperado: 422):
 -> Campo [hora_pedido] com valor [25] respondeu com: 422
 -> Campo [hora_pedido] com valor [-1] respondeu com: 422
 -> Campo [num_itens] com valor [0] respondeu com: 422
 -> Campo [valor_pedido] com valor [-10.0] respondeu com: 422
 -> Campo [distancia_entrega] com valor [-5.0] respondeu com: 422
✅ Todos os testes de contrato estático passaram!


Nesta etapa, focamos na segurança dos dados. Testamos se a API consegue barrar informações inválidas (como horários impossíveis ou valores negativos). Isso protege o nosso modelo de IA de receber lixo e retornar previsões erradas, garantindo que o contrato de dados seja respeitado.

--- 
# SEÇÃO 3 — MLOps: Integração Remota do Modelo Segura e Comportamental 🔒

Aqui validamos o comportamento do modelo baixado do registry. O teste garante que o modelo diferencia perfeitamente um cenário de baixíssimo risco operante de uma anomalia nítida na plataforma.

In [29]:
import numpy as np

def test_modelo_distingue_casos_extremos():
    try:
        from model_utils import load_model
        modelo_remoto = load_model("math04cezario/mlops-bella-tavola-v1")
    except Exception as e:
        print(f"⚠️ Pulando teste comportamental. Erro ao carregar o modelo: {e}")
        return
        
    # Caso 1: Típico, legítimo (R$ 55,00, Almoço às 13h, 2 pratos, sem cancelamentos, perto)
    caso_tipico = np.array([[55.0, 13, 2, 0, 1.5]])
    prob_tipico = modelo_remoto.predict_proba(caso_tipico)[0][1]
    
    # Caso 2: Altíssimo Risco Suspeito (R$ 890,00, Madrugada às 2h, 12 pratos, 4 cancelamentos, super longe)
    caso_suspeito = np.array([[890.0, 2, 12, 4, 45.0]])
    prob_suspeito = modelo_remoto.predict_proba(caso_suspeito)[0][1]
    
    print(f"📊 Probabilidade de Risco - Caso Típico: {prob_tipico:.4f}")
    print(f"📊 Probabilidade de Risco - Caso Suspeito: {prob_suspeito:.4f}")
    
    # Sanidade do comportamento inteligente
    assert prob_suspeito > prob_tipico
    print("✅ O modelo demonstra sanidade de comportamento e discernimento lógico!")

test_modelo_distingue_casos_extremos()

📊 Probabilidade de Risco - Caso Típico: 0.0000
📊 Probabilidade de Risco - Caso Suspeito: 1.0000
✅ O modelo demonstra sanidade de comportamento e discernimento lógico!


/home/codespace/.local/lib/python3.12/site-packages/sklearn/base.py:376: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.6.1 when using version 1.4.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/codespace/.local/lib/python3.12/site-packages/sklearn/base.py:376: InconsistentVersionWarning: Trying to unpickle estimator RandomForestClassifier from version 1.6.1 when using version 1.4.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


Etapa de consolidação do  MLOps. Não testamos apenas se o código roda, mas se a IA 'pensa' corretamente. Comparamos um pedido normal com um pedido suspeito para garantir que o modelo consegue distinguir o risco real, validando o comportamento inteligente da solução vinda do Hugging Face.

--- 
# SEÇÃO 4 — A Estrutura do Pipeline CI/CD (3 Jobs Encadeados) 🚀

Para que o GitHub Actions rode essa suíte inteira de forma correta, o arquivo `.github/workflows/ci.yml` deve conter três macro etapas interdependentes:

```yaml
name: CI — Bella Tavola

on:
  push:
    branches: [main]
  pull_request:
    branches: [main]

jobs:
  # JOB 1: Qualidade e Sanidade Básica
  qualidade:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"
      - name: Instalar dependências
        run: |
          pip install --upgrade pip
          pip install -r requirements.txt
      - name: Formatação estática
        run: black --check .
      - name: Limpeza de Dead Code
        run: autoflake --check --remove-all-unused-imports -r .
      - name: Testes Estruturais Smoke
        run: pytest -v -m smoke

  # JOB 2: Integração e Validação com o Registry (Hugging Face)
  integracao:
    runs-on: ubuntu-latest
    needs: qualidade
    if: github.event_name == 'push' && github.ref == 'refs/heads/main'
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"
      - name: Instalar dependências
        run: pip install -r requirements.txt
      - name: Cache Estratégico do Modelo
        uses: actions/cache@v4
        with:
          path: ~/.cache/huggingface
          key: hf-model-v1-${{ hashFiles('requirements.txt') }}
      - name: Testes Comportamentais Integrados
        run: pytest -v -m integracao --tb=short
        env:
          HF_TOKEN: ${{ secrets.HF_TOKEN }}

  # JOB 3: Relatório Final de Auditoria
  relatorio:
    runs-on: ubuntu-latest
    needs: integracao
    if: github.event_name == 'push' && github.ref == 'refs/heads/main'
    steps:
      - name: Sumário Executivo
        run: |
          echo "================================================"
          echo "  Pipeline CI do Bella Tavola Concluído com Sucesso ✅"
          echo "  Commit ID: ${{ github.sha }}"
          echo "  Autor:     ${{ github.actor }}"
          echo "================================================"
```

Para finalizar, consolidamos tudo em um fluxo automático dividido em três estágios: Qualidade (limpeza de código), Integração (testes com o modelo real) e Relatório. É essa automação que permite que o time foque em desenvolver, enquanto a esteira cuida da segurança.